In [1]:
import pandas as pd
from geographiclib.geodesic import Geodesic
import geopandas
import math
import folium
from shapely.geometry import Point
from shapely.geometry import Polygon
import numpy as np
import matplotlib.pyplot as plt
import random as rand
geod = Geodesic.WGS84 

In [2]:
data = 'obc_sailing.csv'
df = pd.read_csv(data)

In [3]:
print(df.dtypes)

MSG_TYPE                 int64
MMSI                     int64
NAME                    object
IMO_NUMBER             float64
CALL_SIGN               object
LAT_AVG                float64
LON_AVG                float64
PERIOD                  object
SPEED_KNOTS            float64
COG_DEG                float64
HEADING_DEG            float64
NAV_STATUS             float64
NAV_SENSOR             float64
SHIP_AND_CARGO_TYPE      int64
DRAUGHT                float64
DRAUGHT.1              float64
DIM_BOW                  int64
DIM_STERN                int64
DIM_PORT                 int64
DIM_STARBOARD            int64
DESTINATION             object
MMSI_COUNTRY_CD         object
RECEIVER                object
dtype: object


In [4]:
df['PERIOD'] = pd.to_datetime(df['PERIOD'])
print(df['PERIOD'].dtypes)

datetime64[ns]


In [5]:
df = df[['MMSI', 'NAME', 'LAT_AVG', 'LON_AVG', 'PERIOD', 'SPEED_KNOTS', 'COG_DEG', 'HEADING_DEG', 'NAV_STATUS', 'SHIP_AND_CARGO_TYPE', 'DRAUGHT', 'DIM_BOW', 'DIM_STERN', 'DIM_PORT', 'DIM_STARBOARD']]
df['BEAM'] = df['DIM_PORT'] + df['DIM_STARBOARD']
df['LENGTH'] = df['DIM_BOW'] + df['DIM_STERN']
df = df[df['BEAM'] > 0]
df = df[df['LENGTH'] > 0]
df = df[df['DRAUGHT'] > 0]
df = df[df['SPEED_KNOTS'] > 3]
df = df.drop_duplicates(subset=['PERIOD', 'MMSI'])

In [6]:
df['channel_side'] = ['Northwestbound' if x > 200 else 'Southeastbound' for x in df['COG_DEG']]
df = df.sort_values(['MMSI', 'PERIOD']).reset_index(drop=True)
df['time_diff'] = (df.groupby('MMSI')['PERIOD'].diff().dt.total_seconds())
#df['cog_diff'] = (df.groupby('MMSI')['COG_DEG'].diff())
#df['cog_diff'] = np.abs((df['cog_diff'] + 180) % 360 -180)

threshold = 30 * 60
df['new_voyage'] = (df['time_diff'] > threshold) | (df['time_diff'] < 0 ) | (df['time_diff'].isna()) #| (df['cog_diff'] > 75)
df['voyage_id'] = (df.groupby('MMSI')['new_voyage'].cumsum())
df['voyage_id'] = df['voyage_id'].astype('str') + '_' + df['MMSI'].astype('str')
df

,MMSI,NAME,LAT_AVG,LON_AVG,PERIOD,SPEED_KNOTS,COG_DEG,HEADING_DEG,NAV_STATUS,SHIP_AND_CARGO_TYPE,...,DIM_BOW,DIM_STERN,DIM_PORT,DIM_STARBOARD,BEAM,LENGTH,channel_side,time_diff,new_voyage,voyage_id
0,215596000,LADY 8,21.984266,-77.231515,2024-04-02 08:00:00,9.6,321.5,NaN,0.0,36,...,21,8,6,1,7,29,Northwestbound,NaN,True,1_215596000
1,215596000,LADY 8,22.090150,-77.355984,2024-04-02 08:55:00,10.1,308.5,NaN,0.0,36,...,21,8,6,1,7,29,Northwestbound,3300.0,True,2_215596000
2,215596000,LADY 8,22.097433,-77.364399,2024-04-02 09:00:00,10.0,307.6,NaN,0.0,36,...,21,8,6,1,7,29,Northwestbound,300.0,False,2_215596000
3,215596000,LADY 8,22.205648,-77.489949,2024-04-02 09:55:00,10.2,312.1,NaN,0.0,36,...,21,8,6,1,7,29,Northwestbound,3300.0,True,3_215596000
4,215596000,LADY 8,22.210693,-77.496115,2024-04-02 10:00:00,10.2,305.3,NaN,0.0,36,...,21,8,6,1,7,29,Northwestbound,300.0,False,3_215596000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
458,760105000,BAP UNION,22.074000,-77.408113,2024-03-20 15:10:00,5.3,131.5,NaN,3.0,36,...,95,20,6,7,13,115,Southeastbound,300.0,False,4_760105000
459,760105000,BAP UNION,22.071730,-77.405426,2024-03-20 15:15:00,5.0,131.7,NaN,3.0,36,...,95,20,6,7,13,115,Southeastbound,300.0,False,4_760105000
460,760105000,BAP UNION,22.066570,-77.399339,2024-03-20 15:20:00,4.9,132.6,NaN,3.0,36,...,95,20,6,7,13,115,Southeastbound,300.0,False,4_760105000
461,760105000,BAP UNION,22.033813,-77.360017,2024-03-20 15:55:00,5.1,132.7,NaN,3.0,36,...,95,20,6,7,13,115,Southeastbound,2100.0,True,5_760105000


In [7]:
print(df['MMSI'].nunique())
print(df['voyage_id'].nunique())

18
130


In [8]:
data = pd.read_csv('obc_boundaries.csv')
data['geometry'] = data.apply(lambda x: (float(x.Long), float(x.Lat)), axis=1)
channel_coords = list(data['geometry'])
channel_coords_north = [channel_coords[0], channel_coords[1], channel_coords[2], channel_coords[3], channel_coords[-1], channel_coords[-2], channel_coords[-3], channel_coords[-4]]
channel_coords_south = [channel_coords[4], channel_coords[5], channel_coords[6], channel_coords[7], channel_coords[-1], channel_coords[-2], channel_coords[-3], channel_coords[-4]]
coords_north = [channel_coords[0], channel_coords[1], channel_coords[2], (-77.44235, 22.18378333), (-77.44235, 23.80836667), (-78.72296667, 23.80836667)]
coords_south = [channel_coords[4], channel_coords[5], channel_coords[6], channel_coords[7], (-77.4681,21.15348333), (-78.75143333, 21.15348333)]
channel_polygon_n = Polygon(channel_coords_north)
channel_polygon_s = Polygon(channel_coords_south)
polygon_n = Polygon(coords_north)
polygon_s = Polygon(coords_south)

In [9]:
def where_is_vessel(LAT_AVG, LON_AVG, BOOL_northc, BOOL_southc, BOOL_north, BOOL_south):
    if BOOL_northc == True:
        answer = 'in north channel'
    elif BOOL_southc == True:
        answer = 'in south channel'
    elif LON_AVG < -78.72296667:
        answer = 'west'
    elif LON_AVG > -77.49385:
        answer = 'east'
    elif BOOL_north == True:
        answer = 'north'
    elif BOOL_south == True:
        answer = 'south'
    else:
        answer = 'other'
    return answer

In [10]:
geo_df = geopandas.GeoDataFrame(df, geometry=geopandas.points_from_xy(df['LON_AVG'], df['LAT_AVG']), crs='EPSG:4326')
geo_df['in_channel_north'] = geo_df.within(channel_polygon_n)
geo_df['in_channel_south'] = geo_df.within(channel_polygon_s)
geo_df['north_of_channel'] = geo_df.within(polygon_n)
geo_df['south_of_channel'] = geo_df.within(polygon_s)
geo_df['location'] = geo_df.apply(lambda x: where_is_vessel(x.LAT_AVG, x.LON_AVG, x.in_channel_north, x.in_channel_south, x.north_of_channel, x.south_of_channel), axis=1)
geo_df.shape

(463, 27)

In [11]:
geo_df['location'].value_counts()

location
east                171
north               127
west                 90
in south channel     30
in north channel     29
south                16
Name: count, dtype: int64

In [22]:
voyage_list = list(geo_df['voyage_id'].unique())
random_voyages = rand.sample(voyage_list, 40)
geo_df['include'] = geo_df['voyage_id'].isin(random_voyages)
random_voyages_gdf = geo_df[geo_df['include'] == True]
random_voyages_gdf.shape

(87, 28)

In [23]:
data = pd.read_csv('obc_boundaries.csv')
from shapely.geometry import Point
from shapely.geometry import Polygon
data = data[data['Line']!='Middle']
data['geometry'] = data.apply(lambda x: (float(x.Long), float(x.Lat)), axis=1)

channel_coords = list(data['geometry'])
channel_coords_reorder = [channel_coords[0], channel_coords[1], channel_coords[2], channel_coords[3], channel_coords[-1], channel_coords[-2], channel_coords[-3], channel_coords[-4]]
channel_polygon = Polygon(channel_coords_reorder)
geo_df_channel = geopandas.GeoDataFrame(geometry=[channel_polygon], crs='EPSG:4326')

In [24]:
my_map = folium.Map(location=[22.5, -77.8], zoom_start=9.45)
folium.GeoJson(geo_df_channel).add_to(my_map)

def rand_color():
    return "#{:06x}".format(rand.randint(0, 0xFFFFFF))

for id, vessel in random_voyages_gdf.groupby('voyage_id'):
    coords = list(vessel[['LAT_AVG', 'LON_AVG']].itertuples(index=False, name=None))
    if len(coords) < 2:
        continue

    folium.PolyLine(
        coords,
        color=rand_color(),
        weight=2,
        opacity=0.8,
        popup=f"MMSI: {id}"
    ).add_to(my_map)

my_map